# Chapter 3: Methodology - Data Acquisition & Preprocessing

## 3.1 Introduction
This notebook covers the first objective of the research: To acquire and preprocess transcriptomic and genomic datasets from the 1000 Genomes Project and GTEx databases.

We focus on acquiring data for specific candidate genes associated with Neural Tube Defects (NTDs):
- **MTHFR** (Methylenetetrahydrofolate reductase)
- **DHFR** (Dihydrofolate reductase)
- **FOLR1** (Folate receptor alpha)

## 3.2 Tools & Libraries
We utilize `pysam` for efficient handling of VCF files. 
> **Note for Windows Users:** `pysam` handles remote VCF slicing but is difficult to install on Windows. If not installed, this notebook will default to checking connectivity or creating dummy files for testing.

In [3]:
import os
import pandas as pd
import numpy as np
import requests
from io import StringIO

# Handle optional pysam dependency
try:
    import pysam
    PYSAM_AVAILABLE = True
    print("pysam is available. Remote slicing enabled.")
except ImportError:
    PYSAM_AVAILABLE = False
    print("WARNING: pysam not installed. Remote genomic slicing will be unavailable. \nGenerating placeholder data for pipeline testing.")

# Ensure directories exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print("Environment initialized.")

Generating placeholder data for pipeline testing.
Environment initialized.


## 3.3 Genomic Data Acquisition (1000 Genomes Project)

We will access the 1000 Genomes Phase 3 data.

**Gene Locations (GRCh37/hg19):**
- MTHFR: chr1:11,845,709-11,866,160
- DHFR: chr5:79,950,735-80,000,000
- FOLR1: chr11:71,940,000-72,000,000

In [5]:
GENES = {
    'MTHFR': {'chrom': '1', 'start': 11845709, 'end': 11866160},
    'DHFR':  {'chrom': '5', 'start': 79950735, 'end': 79980000}, 
    'FOLR1': {'chrom': '11', 'start': 71600000, 'end': 71620000} 
}

BASE_URL_1KG = "http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/"

def fetch_genotypes(gene_name, chrom, start, end, output_dir='../data/raw'):
    output_file = os.path.join(output_dir, f"{gene_name}_chr{chrom}.vcf")
    
    if PYSAM_AVAILABLE:
        vcf_url = f"{BASE_URL_1KG}ALL.chr{chrom}.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz"
        print(f"Fetching {gene_name} ({chrom}:{start}-{end}) from {vcf_url}...")
        try:
            vcfin = pysam.VariantFile(vcf_url)
            with open(output_file, 'w') as f:
                f.write(str(vcfin.header))
                count = 0
                for record in vcfin.fetch(chrom, start, end):
                    f.write(str(record))
                    count += 1
            print(f"Success: Retrieved {count} variants for {gene_name}.")
            return output_file
        except Exception as e:
            print(f"Error fetching {gene_name}: {e}")
            return None
    else:
        print(f"Mocking download for {gene_name} because pysam is missing...")
        # Create a tiny valid VCF header and a few dummy variants so downstream code doesn't crash
        with open(output_file, 'w') as f:
            f.write("##fileformat=VCFv4.2\n")
            f.write("##source=MockGenerator\n")
            f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSample1\tSample2\n")
            f.write(f"{chrom}\t{start}\t.\tA\tG\t.\tPASS\t.\tGT\t0|0\t0|1\n")
            f.write(f"{chrom}\t{start+100}\t.\tT\tC\t.\tPASS\t.\tGT\t1|1\t0|1\n")
        return output_file

# for gene, coords in GENES.items():
#     fetch_genotypes(gene, coords['chrom'], coords['start'], coords['end'])

## 3.4 Transcriptomic Data Acquisition (GTEx)
Downloads GTEx V8 median gene expression.

In [6]:
def fetch_gtex_median_expression():
    gtex_url = "https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz"
    output_path = "../data/raw/GTEx_median_tpm.gct.gz"
    
    if not os.path.exists(output_path):
        print("Downloading GTEx median expression data...")
        try:
            response = requests.get(gtex_url, stream=True)
            if response.status_code == 200:
                with open(output_path, 'wb') as f:
                    f.write(response.content)
                print("Download complete.")
            else:
                print(f"Failed to download. Status code: {response.status_code}")
        except Exception as e:
            print(f"Request failed: {e}")
    else:
        print("GTEx data already exists locally.")
    
    return output_path

# fetch_gtex_median_expression()